# 49W — Speaker Identification
Maps generic `SPEAKER_00/01/02` labels to real host names using voice embeddings.

**Steps:**
1. For each host, provide a video ID + timestamp where they speak alone
2. Extract their voice embedding (fingerprint)
3. Compare against all diarized videos and relabel speakers by name

**Runtime:** GPU recommended but CPU works fine for this step.

In [ ]:
!pip install -q pyannote.audio torch tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────
AUDIO_DIR    = "/content/drive/MyDrive/49w-bot/data/audio"
DIARIZED_DIR = "/content/drive/MyDrive/49w-bot/data/diarized"
OUTPUT_DIR   = "/content/drive/MyDrive/49w-bot/data/identified"
HF_TOKEN     = "hf_..."  # same token as notebook 02

# For each host: provide a video ID and a time range (seconds)
# where ONLY that host is speaking (e.g., an intro monologue).
# Find these timestamps on YouTube manually.
HOST_SAMPLES = {
    "Host_A": {"video_id": "XXXXXXXXXXX", "start": 10, "end": 40},
    "Host_B": {"video_id": "XXXXXXXXXXX", "start": 10, "end": 40},
    "Host_C": {"video_id": "XXXXXXXXXXX", "start": 10, "end": 40},
}
# ────────────────────────────────────────────────────────────

In [ ]:
import torch
import torchaudio
import numpy as np
from pathlib import Path
from pyannote.audio import Model, Inference

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load speaker embedding model
print("Loading speaker embedding model...")
embedding_model = Model.from_pretrained("pyannote/embedding", use_auth_token=HF_TOKEN)
inference = Inference(embedding_model, window="whole", device=device)

def get_embedding(audio_path: str, start: float, end: float) -> np.ndarray:
    """Extract voice embedding from a specific time range in an audio file."""
    waveform, sample_rate = torchaudio.load(audio_path)
    start_frame = int(start * sample_rate)
    end_frame = int(end * sample_rate)
    clip = waveform[:, start_frame:end_frame]
    # Save temp clip
    tmp = "/tmp/clip.wav"
    torchaudio.save(tmp, clip, sample_rate)
    return np.array(inference(tmp))

# Build voice profile for each host
print("Building voice profiles...")
host_embeddings = {}
for name, sample in HOST_SAMPLES.items():
    audio_path = str(Path(AUDIO_DIR) / f"{sample['video_id']}.wav")
    emb = get_embedding(audio_path, sample["start"], sample["end"])
    host_embeddings[name] = emb
    print(f"  {name}: embedding shape {emb.shape}")

print("Voice profiles ready.")

In [ ]:
import json
from tqdm.notebook import tqdm
from pyannote.audio import Inference as Inf
from scipy.spatial.distance import cosine

def cosine_similarity(a, b):
    return 1 - cosine(a, b)

def identify_speaker(speaker_embedding: np.ndarray, host_embeddings: dict, threshold: float = 0.7) -> str:
    """Match a speaker embedding to the closest known host. Returns 'Unknown' if below threshold."""
    best_name, best_score = "Unknown", 0.0
    for name, emb in host_embeddings.items():
        score = cosine_similarity(speaker_embedding, emb)
        if score > best_score:
            best_score, best_name = score, name
    return best_name if best_score >= threshold else "Unknown"

diarized_files = sorted(Path(DIARIZED_DIR).glob("*.json"))
print(f"Processing {len(diarized_files)} diarized files...")

for diarized_path in tqdm(diarized_files, desc="Identifying speakers"):
    vid_id = diarized_path.stem
    out_file = Path(OUTPUT_DIR) / f"{vid_id}.json"

    if out_file.exists():
        continue

    with open(diarized_path, encoding="utf-8") as f:
        record = json.load(f)

    audio_path = str(Path(AUDIO_DIR) / f"{vid_id}.wav")
    if not Path(audio_path).exists():
        continue

    # Get embedding for each unique speaker in this video
    speaker_labels = {}
    for speaker_id in record["speakers"]:
        # Collect all turns for this speaker, use longest turn for embedding
        turns = [t for t in record["turns"] if t["speaker"] == speaker_id]
        longest = max(turns, key=lambda t: t["end"] - t["start"])
        duration = longest["end"] - longest["start"]
        if duration < 3:  # skip very short turns
            speaker_labels[speaker_id] = "Unknown"
            continue
        emb = get_embedding(audio_path, longest["start"], longest["end"])
        speaker_labels[speaker_id] = identify_speaker(emb, host_embeddings)

    # Relabel turns with real names
    for turn in record["turns"]:
        turn["speaker"] = speaker_labels.get(turn["speaker"], "Unknown")

    record["speaker_mapping"] = speaker_labels

    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False, indent=2)

print("Speaker identification complete.")

In [ ]:
# Show speaker turn distribution across all videos
from collections import Counter

speaker_words = Counter()
for path in Path(OUTPUT_DIR).glob("*.json"):
    with open(path) as f:
        rec = json.load(f)
    for turn in rec["turns"]:
        speaker_words[turn["speaker"]] += len(turn["text"].split())

print("Words per speaker across all videos:")
for speaker, words in speaker_words.most_common():
    print(f"  {speaker}: {words:,} words")